#### EDA by using Pyspark

In [1]:
from pyspark.sql import SparkSession
from pyspark.sql.functions import *

In [2]:
spark = SparkSession.builder.appName("student").getOrCreate()

In [3]:
df = spark.read.csv("student_performance_updated_1000.csv", header = True, sep =",", inferSchema = True)
df.show()

+---------+-----------------+------+--------------+-----------------+-------------+-------------------------+---------------+----------+-----------+--------------+--------------------+
|StudentID|             Name|Gender|AttendanceRate|StudyHoursPerWeek|PreviousGrade|ExtracurricularActivities|ParentalSupport|FinalGrade|Study Hours|Attendance (%)|Online Classes Taken|
+---------+-----------------+------+--------------+-----------------+-------------+-------------------------+---------------+----------+-----------+--------------+--------------------+
|      1.0|             John|  Male|          85.0|             15.0|         78.0|                      1.0|           High|      80.0|        4.8|          59.0|               false|
|      2.0|            Sarah|Female|          90.0|             20.0|         85.0|                      2.0|         Medium|      87.0|        2.2|          70.0|                true|
|      3.0|             Alex|  Male|          78.0|             10.0|      

In [4]:
# analisa statistic
df.describe().show()

+-------+-----------------+--------------+------+-----------------+------------------+------------------+-------------------------+---------------+-----------------+------------------+------------------+
|summary|        StudentID|          Name|Gender|   AttendanceRate| StudyHoursPerWeek|     PreviousGrade|ExtracurricularActivities|ParentalSupport|       FinalGrade|       Study Hours|    Attendance (%)|
+-------+-----------------+--------------+------+-----------------+------------------+------------------+-------------------------+---------------+-----------------+------------------+------------------+
|  count|              960|           966|   952|              960|               950|               967|                      957|            978|              960|               976|               959|
|   mean|5416.019791666667|          NULL|  NULL|85.51041666666667|17.630526315789474| 77.59875904860392|       1.5203761755485894|           NULL|80.03020833333333|2.4069672131147564|

In [5]:
df.summary().show()

+-------+-----------------+--------------+------+-----------------+------------------+------------------+-------------------------+---------------+-----------------+------------------+------------------+
|summary|        StudentID|          Name|Gender|   AttendanceRate| StudyHoursPerWeek|     PreviousGrade|ExtracurricularActivities|ParentalSupport|       FinalGrade|       Study Hours|    Attendance (%)|
+-------+-----------------+--------------+------+-----------------+------------------+------------------+-------------------------+---------------+-----------------+------------------+------------------+
|  count|              960|           966|   952|              960|               950|               967|                      957|            978|              960|               976|               959|
|   mean|5416.019791666667|          NULL|  NULL|85.51041666666667|17.630526315789474| 77.59875904860392|       1.5203761755485894|           NULL|80.03020833333333|2.4069672131147564|

In [6]:
#common utility functions
#tambah kolom baru attendance di bagi 100 (bukan percen)

df = df.withColumn("Attendance (not %)", col("Attendance (%)")/100)
df.show(5, False)

+---------+-------+------+--------------+-----------------+-------------+-------------------------+---------------+----------+-----------+--------------+--------------------+------------------+
|StudentID|Name   |Gender|AttendanceRate|StudyHoursPerWeek|PreviousGrade|ExtracurricularActivities|ParentalSupport|FinalGrade|Study Hours|Attendance (%)|Online Classes Taken|Attendance (not %)|
+---------+-------+------+--------------+-----------------+-------------+-------------------------+---------------+----------+-----------+--------------+--------------------+------------------+
|1.0      |John   |Male  |85.0          |15.0             |78.0         |1.0                      |High           |80.0      |4.8        |59.0          |false               |0.59              |
|2.0      |Sarah  |Female|90.0          |20.0             |85.0         |2.0                      |Medium         |87.0      |2.2        |70.0          |true                |0.7               |
|3.0      |Alex   |Male  |78.0

In [7]:
# filter attendance lebih dari 80

df_filtered = df.filter(col("Attendance (%)") > 80)
df_filtered.show(5)

+---------+-------+------+--------------+-----------------+-------------+-------------------------+---------------+----------+-----------+--------------+--------------------+------------------+
|StudentID|   Name|Gender|AttendanceRate|StudyHoursPerWeek|PreviousGrade|ExtracurricularActivities|ParentalSupport|FinalGrade|Study Hours|Attendance (%)|Online Classes Taken|Attendance (not %)|
+---------+-------+------+--------------+-----------------+-------------+-------------------------+---------------+----------+-----------+--------------+--------------------+------------------+
|      3.0|   Alex|  Male|          78.0|             10.0|         65.0|                      0.0|            Low|      68.0|        4.6|          92.0|               false|              0.92|
|      4.0|Michael|  Male|          92.0|             25.0|         90.0|                      3.0|           High|      92.0|        2.9|          96.0|               false|              0.96|
|      5.0|   Emma|Female|    

In [8]:
#Filtering kolom yang muncul (tentukan kolom)
df_selected = df_filtered.select("StudentID", "Name", "Gender", "FinalGrade","Attendance (%)")
df_selected.show(5) #liat hasil

+---------+-------+------+----------+--------------+
|StudentID|   Name|Gender|FinalGrade|Attendance (%)|
+---------+-------+------+----------+--------------+
|      3.0|   Alex|  Male|      68.0|          92.0|
|      4.0|Michael|  Male|      92.0|          96.0|
|      5.0|   Emma|Female|      85.0|          97.0|
|      6.0| Olivia|Female|      NULL|          97.0|
|      7.0| Daniel|  Male|      62.0|          96.0|
+---------+-------+------+----------+--------------+
only showing top 5 rows



In [9]:
missing_val = df.filter(col("StudentID").isNull() | isnan("StudentID") | (col("StudentID") == "")).count()
missing_val

40

In [10]:
#Handling missing value
#Hapus studentid yang kosong

df_clean = df.na.drop(subset = ["StudentID"])

missing_val = df_clean.filter(col("StudentID").isNull() | isnan("StudentID") | (col("StudentID") == "")).count()
missing_val

0

In [11]:
#misalnya isi kolom yang studentID dengan 0
df_filled = df.fillna({"StudentID":0})

In [12]:
print("Data yang sudah dibersihkan (drop null) :")
df_clean.show(5, False)

Data yang sudah dibersihkan (drop null) :
+---------+-------+------+--------------+-----------------+-------------+-------------------------+---------------+----------+-----------+--------------+--------------------+------------------+
|StudentID|Name   |Gender|AttendanceRate|StudyHoursPerWeek|PreviousGrade|ExtracurricularActivities|ParentalSupport|FinalGrade|Study Hours|Attendance (%)|Online Classes Taken|Attendance (not %)|
+---------+-------+------+--------------+-----------------+-------------+-------------------------+---------------+----------+-----------+--------------+--------------------+------------------+
|1.0      |John   |Male  |85.0          |15.0             |78.0         |1.0                      |High           |80.0      |4.8        |59.0          |false               |0.59              |
|2.0      |Sarah  |Female|90.0          |20.0             |85.0         |2.0                      |Medium         |87.0      |2.2        |70.0          |true                |0.7     

In [13]:
print("Data yang sudah diisi nilai default :\n")
df_filled.show(5)

Data yang sudah diisi nilai default :

+---------+-------+------+--------------+-----------------+-------------+-------------------------+---------------+----------+-----------+--------------+--------------------+------------------+
|StudentID|   Name|Gender|AttendanceRate|StudyHoursPerWeek|PreviousGrade|ExtracurricularActivities|ParentalSupport|FinalGrade|Study Hours|Attendance (%)|Online Classes Taken|Attendance (not %)|
+---------+-------+------+--------------+-----------------+-------------+-------------------------+---------------+----------+-----------+--------------+--------------------+------------------+
|      1.0|   John|  Male|          85.0|             15.0|         78.0|                      1.0|           High|      80.0|        4.8|          59.0|               false|              0.59|
|      2.0|  Sarah|Female|          90.0|             20.0|         85.0|                      2.0|         Medium|      87.0|        2.2|          70.0|                true|           

In [14]:
# checking missing value
df_filled.filter((col("StudentID").isNull() | isnan("StudentID"))).count()

0

In [15]:
df_clean.count() #sisa dari studentID yang null uda di hapus semua

960

In [16]:
#bersihin semua missing value di kolom dengan drop na
df_drop = df.na.drop()

In [17]:
df_drop.count()

645

In [18]:
df_drop = df_drop.withColumn("Online Classes Taken", col("Online Classes Taken").cast("int"))

In [19]:
df_drop.dtypes

[('StudentID', 'double'),
 ('Name', 'string'),
 ('Gender', 'string'),
 ('AttendanceRate', 'double'),
 ('StudyHoursPerWeek', 'double'),
 ('PreviousGrade', 'double'),
 ('ExtracurricularActivities', 'double'),
 ('ParentalSupport', 'string'),
 ('FinalGrade', 'double'),
 ('Study Hours', 'double'),
 ('Attendance (%)', 'double'),
 ('Online Classes Taken', 'int'),
 ('Attendance (not %)', 'double')]

In [20]:
col_num = ["StudentID", "AttendanceRate", "PreviousGrade", "ExtracurricularActivities", "FinalGrade", "Study Hours", "Attendance (%)", "Attendance (not %)", "Online Classes Taken"]

In [24]:
df_drop.select([
    count(when(col(c).isNull()|isnan(c),c)).alias(c) for c in col_num
]).show() #di data ini angka 0 dikolom numerik tidak dihapus

+---------+--------------+-------------+-------------------------+----------+-----------+--------------+------------------+--------------------+
|StudentID|AttendanceRate|PreviousGrade|ExtracurricularActivities|FinalGrade|Study Hours|Attendance (%)|Attendance (not %)|Online Classes Taken|
+---------+--------------+-------------+-------------------------+----------+-----------+--------------+------------------+--------------------+
|        0|             0|            0|                        0|         0|          0|             0|                 0|                   0|
+---------+--------------+-------------+-------------------------+----------+-----------+--------------+------------------+--------------------+



In [27]:
# check missing value di kolom string
col_cat  = ["Name", "Gender", "ParentalSupport"]

In [28]:
df_drop.select([
    count(when(col(c).isNull()|isnan(c)|(col(c) == ""),c)).alias(c) for c in col_cat
]).show()

+----+------+---------------+
|Name|Gender|ParentalSupport|
+----+------+---------------+
|   0|     0|              0|
+----+------+---------------+

